_We'll build this together, live, using everything from this session — dictionaries, sets, exceptions, and reading from files._

## Application: Cleaning a Trade Log

A trading desk logs every trade to a text file, one trade per line, as `ticker,quantity,price`:

```
AAPL,150,192.34
MSFT,80,310.5
```

But data feeds glitch. Some lines in today's log are corrupted — a missing field, a price that isn't a number, or a line that's garbled entirely. We want to:

1. Skip corrupted lines without crashing.
2. Total up the quantity traded per ticker, for the lines that *are* valid.

## Part 1: Total quantity by ticker

**Step 1: read the file, line by line.** We open the file and loop through it — each `line` is one trade record (as a string, including corrupted ones).

**Hint:** open the file with `with open(filepath) as f:`, then loop `for line in f:` to get one line at a time:

In [11]:
with open('data/trade_log.txt') as f:
    for line in f:
        print(repr(line))
        break  # just show the first line

'MSFT,11,312.42\n'


**Hint:** each line from a file is a string ending in a newline -- `.strip()` removes it:

In [8]:
example_lines = ["AAPL,150,192.34\n", "MSFT,80,310.5\n"]
for line in example_lines:
    print(repr(line.strip()))

'AAPL,150,192.34'
'MSFT,80,310.5'


In [38]:
# def total_quantity_by_ticker(filepath):
#     """Return a dict mapping ticker to total quantity traded, skipping corrupted lines."""
#     trade_dict = {}
#     with open(filepath) as f:
#         for line in f:
#             parts = line.strip().split(',')
#             trade_dict[ticker], trade_dict[quantity], trade_dict[price] = line.strip().split(',')
#         return trade_dict
# total_quantity_by_ticker('data/trade_log.txt')


def total_quantity_by_ticker(filepath):
    corrupt_lines = 0
    trades = 0
    quantity_traded = 0
    value_traded = 0
    unique_tickers = set() #SET!
                            #list quadratic complexity
                            #dict & set?
    summary ={}
    
    with open(filepath) as f:
        for line in f:
            
            try:
                ticker, qty_str, price_str = line.strip().split(',')
                qty = int(qty_str)
                price = float(price_str)
                unique_tickers.add(ticker)
                
                if ticker in summary:
                    summary[ticker][trades] +=1
                    summary[ticker][quantity] += qty
                    summary[ticker][value] += qty*price
                else:
                    summary[ticker] = {'trades' : 1, 'quantity' : qty, 'value': round(price*qty, 4) }
                
            except Exception as e:
                print(e)
    return summary
        
total_quantity_by_ticker('data/trade_log.txt')

0
0
invalid literal for int() with base 10: ''
0
could not convert string to float: 'N/A'
0
0
not enough values to unpack (expected 3, got 1)
0
0
0
0
could not convert string to float: ''
0
0
0
could not convert string to float: '25.3.76'
0
0
0
invalid literal for int() with base 10: 'abc'
0
0
0
not enough values to unpack (expected 3, got 1)
0
0
0
0
too many values to unpack (expected 3)
0
0
not enough values to unpack (expected 3, got 2)
0
0
0
0


{'MSFT': {'trades': 1, 'quantity': 11, 'value': 3436.62},
 'AMZN': {'trades': 1, 'quantity': 62, 'value': 10936.8},
 'AAPL': {'trades': 1, 'quantity': 28, 'value': 5241.32},
 'META': {'trades': 1, 'quantity': 61, 'value': 29827.78},
 'TSLA': {'trades': 1, 'quantity': 6, 'value': 1531.08},
 'GOOGL': {'trades': 1, 'quantity': 183, 'value': 26238.54},
 'NFLX': {'trades': 1, 'quantity': 31, 'value': 18657.35},
 'NVDA': {'trades': 1, 'quantity': 140, 'value': 17327.8}}

**Step 2: parse each line, and catch the bad ones.** Split the line on `,` and unpack it into three parts, then convert quantity and price to numbers. Any of these can fail — too few parts, or text that isn't a number — and either failure raises a `ValueError`. Wrap it in `try`/`except` so one bad line doesn't crash the whole loop.

**Hint:** `.split(",")` breaks a line into its comma-separated parts, which you can unpack straight into three variables:

In [9]:
ticker, qty_str, price_str = "AAPL,150,192.34".split(",")
print(ticker, qty_str, price_str)

# What if a line has a missing or extra column?
print("AAPL,150".split(","))               # only 2 parts
print("AAPL,150,192.34,extra".split(","))  # 4 parts

try:
    ticker, qty_str, price_str = "AAPL,150".split(",")
except ValueError as e:
    print("Unpacking failed:", e)

AAPL 150 192.34
['AAPL', '150']
['AAPL', '150', '192.34', 'extra']
Unpacking failed: not enough values to unpack (expected 3, got 2)


**Hint:** converting text that isn't a number raises a `ValueError` -- this is exactly what `try`/`except` catches:

In [10]:
try:
    int("N/A")           # a quantity logged as "N/A" instead of a number
except ValueError:
    print("Bad quantity!")

try:
    float("192.34.56")   # a garbled price -- two decimal points
except ValueError:
    print("Bad price!")

Bad quantity!
Bad price!


In [ ]:
def total_quantity_by_ticker(filepath):
    """Return a dict mapping ticker to total quantity traded, skipping corrupted lines."""
    # your code here


**Step 3: aggregate into a dictionary.** For each valid line, add its quantity to that ticker's running total — using `dict.get(key, 0)` to handle the first time we see a ticker.

**Hint:** `dict.get(key, 0)` returns the current total for a ticker, or `0` if it's not in the dictionary yet -- no need to check first:

In [28]:
totals = {"AAPL": 150}
print(totals.get("AAPL", 0))
print(totals.get("MSFT", 0))

150
0


In [ ]:
def total_quantity_by_ticker(filepath):
    """Return a dict mapping ticker to total quantity traded, skipping corrupted lines."""
    # your code here


**Now test it on today's log:**

In [ ]:
totals = total_quantity_by_ticker('data/trade_log.txt')
print(f"Total quantity by ticker: {totals}")

Total quantity by ticker: {'MSFT': {'trades': 1, 'quantity': 11, 'value': 3436.62}, 'AMZN': {'trades': 1, 'quantity': 62, 'value': 10936.8}, 'AAPL': {'trades': 1, 'quantity': 28, 'value': 5241.32}, 'META': {'trades': 1, 'quantity': 61, 'value': 29827.78}, 'TSLA': {'trades': 1, 'quantity': 6, 'value': 1531.08}, 'GOOGL': {'trades': 1, 'quantity': 183, 'value': 26238.54}, 'NFLX': {'trades': 1, 'quantity': 31, 'value': 18657.35}, 'NVDA': {'trades': 1, 'quantity': 140, 'value': 17327.8}}


## Part 2: Your turn: which tickers actually traded?

The desk also wants a quick list of every ticker that appears in a *valid* trade today — no totals, just which ones showed up, with no duplicates.

Write a function `distinct_tickers(filepath)` that returns a **set** of every ticker found in valid lines of the log (corrupted lines still don't count). It follows the same shape as `total_quantity_by_ticker` — read the file, parse each line inside a `try`/`except`, skip failures — but instead of accumulating a total, add each valid ticker to a set.

**Hint:** `set.add(...)` adds an item, and adding the same item twice has no effect -- a set never has duplicates:

In [36]:
tickers = set()
tickers.add("AAPL")
tickers.add("AAPL")
tickers.add("MSFT")
tickers

{'AAPL', 'MSFT'}

In [ ]:
def distinct_tickers(filepath):
    tickers = set()
    with open(filepath) as file:
        for line in file:
            try:
                ticker, _, _2 = line.strip().split(',')
                tickers.add(ticker)
            except Exception as e:
                continue    
    return tickers

distinct_tickers('data/trade_log.txt')

{'AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NFLX', 'NVDA', 'TSLA'}

## Part 3: Your turn: total value traded

The desk also wants a single number: the total dollar value traded today, across every valid trade (`quantity * price`, summed over all tickers).

Write a function `total_value(filepath)` that returns this total as a float. It follows the same shape as before, but this time there's no dictionary or set involved -- just one running total.

In [37]:
def distinct_tickers(filepath):
    tickers = set()
    total_value_traded = 0
    with open(filepath) as file:
        for line in file:
            try:
                ticker, qty, price = line.strip().split(',')
                qty = int(qty)
                price = float(price)
                tickers.add(ticker)
                total_value_traded += price*qty
            except Exception as e:
                continue    
    return tickers, total_value_traded

distinct_tickers('data/trade_log.txt')

({'AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NFLX', 'NVDA', 'TSLA'}, 1027966.99)

## Part 4: Your turn: how are lines corrupted?

Knowing *how many* lines failed isn't as useful as knowing *why*. Categorize each corrupted line into one of:

- `"wrong number of fields"`: splitting the line on `,` doesn't give exactly 3 parts
- `"bad number"`: there are 3 parts, but the quantity or price isn't a valid number

Write a function `corrupted_line_counts(filepath)` that returns a dictionary mapping each category to how many lines fell into it, e.g. `{"wrong number of fields": 2, "bad number": 3}`.

**Hint:** check `len(parts)` right after splitting, before unpacking -- that sorts a line into a category *before* you'd hit a `ValueError` from unpacking the wrong number of parts:

In [ ]:
# your code here